# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Japhetthefirst/flyrank_ml_intership/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking.**

The goal is to decide which pages a reviewer should look at first. The output is an ordered queue of pages, each with a score, a reason code, and a suggested action.

I may use a classifier's probability of "declining" to build the score, but what I deliver and judge is the order of the list (Ranking), not a yes/no for each page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Japhetthefirst/flyrank_ml_intership"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not IN_COLAB:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape[0], "pages,", df.shape[1], "columns")

Loaded: 30000 pages, 44 columns


## 2. Target or proxy

**What I predict:** for each page that already has search exposure, whether it is declining: `is_declining_label` (1 = declining, 0 = not). I then sort pages by that score to build the review queue. The queue is the output, and the label is what the model learns to score.

**Where the label comes from: a defined rule, not an observed future outcome.**
- `trend_pct` is the % change in impressions over the last 30 days compared with the 30 days before.
- `trend_direction` buckets that number: "down" means trend_pct is -20% or lower, "stable" is between -20% and +20%, and "up" is +20% or higher. "New" (no impressions in the earlier window) and "flat" (no impressions in either window) have no trend_pct.
- `is_declining_label` = 1 when trend_direction is "down", otherwise 0.

**This is a proxy, and I'll say so.** It describes what already happened in one 30-day comparison. It doesn't say what happens next, or whether a refresh would help. The "not declining" group (0) is also mixed: stable, growing, new, and no-data pages.

**Guardrail:** `trend_pct` and `trend_direction` are never used as model inputs, because the label is built from them. Using them would be leakage.

> **Later improvement:** a future-window label (features from the prior 90 days, outcome in the next 30 days), if the warehouse data supports it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the proxy label and check that the rule is what I think it is
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(df["is_declining_label"].value_counts().rename({1: "declining (1)", 0: "not declining (0)"}))
print("Declining rate:", round(df["is_declining_label"].mean() * 100, 1), "%")

down = df[df["trend_direction"] == "down"]["trend_pct"]
print("\nIn the 'down' group, trend_pct ranges from", down.min(), "to", down.max())

print("\nWhat's inside the 'not declining' group:")
print(df[df["is_declining_label"] == 0]["trend_direction"].value_counts())

is_declining_label
declining (1)        16262
not declining (0)    13738
Name: count, dtype: int64
Declining rate: 54.2 %

In the 'down' group, trend_pct ranges from -100.0 to -20.0

What's inside the 'not declining' group:
trend_direction
stable    5962
up        4388
new       2236
flat      1152
Name: count, dtype: int64


## 3. Success metric

**Metric: Precision@50, measured on clients the model has never seen.**

Of the 50 pages my ranking puts first, how many are actually labelled declining (`is_declining_label` = 1)? I chose K = 50 because that is how many pages I assumed a reviewer can check in one cycle (Section 2 of Week 1). Holding out whole clients means a model can't win by recognising a client it already saw.

**What "good" means, in order:**
1. **Beat random.** About 54% of pages are labelled declining, so picking 50 pages at random gives roughly 27 right. Any ranking has to clearly beat that.
2. **Beat my own baseline rule**. If a learned model can't beat a simple, readable rule, the rule is the better answer, and I'll report that.

Because the label is a proxy, a high Precision@50 means "my ranking agrees with the declining rule", not "these pages will recover if refreshed".

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values
base_rate = y.mean()
print(f"Base rate (what random picking gets): {base_rate:.3f}  -> about {round(base_rate * 50)} of 50")

rng = np.random.default_rng(42)
random_p50 = np.mean([precision_at_k(rng.random(len(y)), y, 50) for _ in range(500)])
print(f"Random ranking, Precision@50 (average of 500 tries): {random_p50:.3f}")

print(f"Starter reference (from the repo's own run): random forest 0.740 -> about {round(0.74 * 50)} of 50")

Base rate (what random picking gets): 0.542  -> about 27 of 50
Random ranking, Precision@50 (average of 500 tries): 0.537
Starter reference (from the repo's own run): random forest 0.740 -> about 37 of 50


## 4. The unit of analysis, as a real dataframe

**One row = one page** (one `content_id`), with its trailing 90-day numbers. The lane slice follows the starter pipeline's rules: pages with impressions_90d > 0 and content_age_days >= 90, one row per content_id. In the starter data that keeps all 30,000 pages (32 clients).

**Column roles:**
- **Identifiers (never model inputs):** `content_id`, `client_id`. I use `client_id` only to split train and test by client.
- **Candidate features (known before the outcome):** `content_type`, `word_count`, `impressions_90d`, `ctr`, `avg_position`, `days_since_last_update`.
- **Label source (shown for checking, never a feature):** `trend_pct`, `trend_direction`.
- **Target (proxy):** `is_declining_label` = 1 when `trend_direction` is "down".

**Data gotchas I'll handle later:** `ctr` is a percentage (0.76 means 0.76%). `avg_position` = 0 means "no position data" (1,205 pages), not rank zero. `word_count` is blank for 7,699 pages, so I'll add a "has word count" flag instead of silently filling zeros. `trend_pct` is blank for 3,388 pages (new or flat), which is another reason it can't be a feature.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Lane 2 slice: same rules as the starter pipeline's prepare step
lane = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
print("Lane slice:", lane.shape[0], "rows (one row = one page) from", df.shape[0], "raw rows")
print("Unique content_id:", lane["content_id"].is_unique, "| clients:", lane["client_id"].nunique())

show_cols = ["content_id", "client_id",                                      # identifiers
             "content_type", "word_count", "impressions_90d", "ctr",         # candidate features
             "avg_position", "days_since_last_update",
             "trend_pct", "trend_direction",                                 # label source (not features)
             "is_declining_label"]                                           # target (proxy)
lane[show_cols].head(5)

Lane slice: 30000 rows (one row = one page) from 30000 raw rows
Unique content_id: True | clients: 32


,content_id,client_id,content_type,word_count,impressions_90d,ctr,avg_position,days_since_last_update,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,3803,0.76,10.6,20,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,15320,0.05,20.3,25,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,12581,0.09,36.5,20,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,11751,0.49,6.2,22,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,19140,0.13,44.0,14,-34.7,down,1


In [5]:
# The gotchas I wrote about, counted
print("word_count blank:", lane["word_count"].isna().sum())
print("avg_position == 0 (no data):", (lane["avg_position"] == 0).sum())
print("trend_pct blank (new/flat):", lane["trend_pct"].isna().sum())

word_count blank: 7699
avg_position == 0 (no data): 1205
trend_pct blank (new/flat): 3388


## 5. Why ML beats a fixed rule here

**Why a learned ranking *might* beat a fixed rule here:** many signals are involved (visibility, position, CTR, engagement, age, length, page type), and they overlap. A single if-statement ends up covering only a slice of the data, and it doesn't show how the signals combine.

**What the starter data shows (observed, directional):**
- "Stale and visible" matches only **17** of 30,000 pages. It's very precise (16 of those 17 are declining) but covers almost nothing.
- The other readable rules each cover thousands of pages (for example 9,759 for "low CTR on a visible page"), but they are only a little better than the 54.2% base rate: 62.7% declining for low-CTR pages, and about 52% to 55% for the others.
- The rules overlap (6,076 pages match two or more), and stacking them is not a clean signal: pages matching 0, 1, 2, and 3 rules are declining 49.8%, 58.1%, 59.4%, and 49.2% of the time. "More rules matched" does not mean "more at risk".

That's the kind of messy, tangled pattern where a learned ranking could help, because it can weigh signals together instead of counting rule hits.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.